# Notebook 18. Ising onramp benchmark: PyMatching baseline

Companion to Notebook 17 (the NVIDIA Ising integration walkthrough).

Before you can claim a pre-decoder is buying you anything, you need a
fair MWPM baseline at the same `(distance, rounds, p_error, basis)`
you plan to evaluate the pre-decoder on. This notebook plots the
baseline sweep that ships with the repo (`benchmarks/ising/pymatching_sweep.json`)
so you can see what the residual-MWPM half of the Ising chain has
to clear before its worth running.

What you should see:

- LER drops with distance at every `p_error` (the surface code is
  doing its job and we're below threshold).
- LER rises steeply with `p_error` (also expected).
- Curves separate cleanly, anything inside the standard error bars
  is noise, anything outside is signal a pre-decoder can compete with.

In [ ]:
from pathlib import Path
import json

sweep_path = Path('../benchmarks/ising/pymatching_sweep.json')
data = json.loads(sweep_path.read_text())
print('Shots per point:', data['metadata']['shots'])
print('Total points  :', len(data['results']))
data['results'][0]

## LER vs physical error rate, grouped by code distance

X basis only here. Z is symmetric for the rotated surface code under
this circuit-level depolarising noise model.

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

by_distance = defaultdict(list)
for r in data['results']:
    if r['basis'] != 'X':
        continue
    by_distance[r['distance']].append(r)

fig, ax = plt.subplots(figsize=(7, 5))
for d in sorted(by_distance):
    pts = sorted(by_distance[d], key=lambda r: r['p_error'])
    p = [r['p_error'] for r in pts]
    ler = [r['rate'] for r in pts]
    se  = [r['standard_error'] for r in pts]
    ax.errorbar(p, ler, yerr=se, marker='o', capsize=3, label=f'd={d}')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate (PyMatching MWPM)')
ax.set_title('Rotated surface code. PyMatching baseline\n(rounds = distance, X basis, 5K shots/point)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(title='Code distance')
plt.tight_layout()
plt.show()

## What this gives you for the Ising pre-decoder run

The numbers in the chart above are what the NVIDIA Ising pre-decoder
(when wrapped in `IsingDecoderWrapper` from Notebook 17) has to beat.
Specifically:

1. Pick a `(distance, p_error)` cell where the baseline LER is large
   enough to measure with reasonable statistics, at 5K shots the
   `d=3, p=0.005` cell is comfortable, `d=7, p=0.001` is not.
2. Run the same `(d, p, rounds, basis)` through the wrapper.
3. Compare the rates with their standard errors. A pre-decoder is
   only worth its compute if the residual rate is below the baseline
   by more than the combined SE.

If you want a tighter sweep:

```bash
python -m benchmarks.ising.run_pymatching_sweep \
    --distances 3 5 7 9 \
    --p-errors 0.0005 0.001 0.002 0.003 0.005 \
    --shots 20000 \
    --output benchmarks/ising/pymatching_sweep_extended.json
```

Hardware data (Fez surface patches) will land here once we've fed it
through, it goes in the same JSON shape so this notebook plots it
without changes.